In [1]:
import pandas as pd
import os
import networkx as nx
from tqdm import tqdm

In [2]:
#ad_dir_path = '../../../data_preprocessed/grn_output/ad_nci_seperate_network/lioness/ad_preprocessed/'
nci_dir_path = '../../../data_preprocessed/grn_output/ad_nci_seperate_network/lioness/nci_preprocessed/'

In [3]:
networks_list_nci = []

for i in tqdm(os.listdir(nci_dir_path)):
    network = pd.read_csv(nci_dir_path+i)
    remove_genes = list(set(network['0']).intersection(network['1']))
    network = network[~network['1'].isin(remove_genes)]
    B = nx.Graph()
    B.add_nodes_from(network['0'].tolist(), bipartite = 'tf')
    B.add_nodes_from(network['1'].tolist(), bipartite = 'target_genes')
    for k,i in network.iterrows():
        #print()
        #print(i['1'])
        #print(i['2'])
        B.add_edge(i['0'],i['1'],weight = i['2'])

    networks_list_nci.append(B)
        


100%|███████████████████████████████████████████| 67/67 [11:24<00:00, 10.22s/it]


In [ ]:
import networkx as nx
import numpy as np
import glob

# Lists to store network metrics
ad_metrics = []
nci_metrics = []

def compute_network_metrics(G):
    """
    Compute various network properties for a given weighted bipartite gene regulatory network.
    """
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    
    # Calculate density
    regulators = {n for n, d in G.nodes(data=True) if d.get("bipartite", 0) == 'tf'}
    genes = set(G) - regulators
    max_possible_edges = len(regulators) * len(genes)
    density = num_edges / max_possible_edges if max_possible_edges > 0 else 0
    
    # Weighted Average degree
    avg_degree = (2 * sum(d for _, d in G.degree(weight='weight'))) / num_nodes if num_nodes > 0 else 0
    
    # Modularity (requires community detection, using greedy modularity method)
    from networkx.algorithms import community
    communities = list(community.greedy_modularity_communities(G))
    modularity = community.modularity(G, communities, weight='weight') if communities else 0
    
    # Weighted Clustering coefficient (bipartite version)
    clustering = nx.average_clustering(G, weight='weight')
    
    # Giant component size
    largest_cc = max(nx.connected_components(G), key=len, default=set())
    giant_component_size = len(largest_cc)
    
    # Weighted Nestedness (proxy: assortativity coefficient)
    nestedness = nx.degree_assortativity_coefficient(G, weight='weight') if num_nodes > 1 else 0
    
    return [density, avg_degree, modularity, clustering, giant_component_size, nestedness]



# Load all Control networks and compute metrics


for network_nci in tqdm(networks_list_nci):
    nci_metrics.append(compute_network_metrics(network_nci))



 97%|████████████████████████████████████▊ | 65/67 [30:45:27<40:45, 1222.81s/it]

In [7]:
# Convert results to NumPy arrays for further analysis
#ad_metrics = np.array(ad_metrics)
nci_metrics = np.array(nci_metrics)

# Print sample output
#print("AD Metrics (first 5 networks):\n", ad_metrics[:5])
print("Control Metrics (first 5 networks):\n", nci_metrics[:5])

Control Metrics (first 5 networks):
 [[ 4.84857828e-02  7.73967979e+02  1.99310664e-01  0.00000000e+00
   1.71820000e+04 -4.88543290e-01]
 [ 5.08207215e-02  7.47584168e+02  1.97717780e-01  0.00000000e+00
   1.71780000e+04 -5.37919611e-01]
 [ 5.05410165e-02  7.94731633e+02  1.91119233e-01  0.00000000e+00
   1.72030000e+04 -4.88686063e-01]
 [ 5.05034277e-02  7.51329213e+02  2.10921766e-01  0.00000000e+00
   1.71860000e+04 -5.28289944e-01]
 [ 4.94248400e-02  7.45736438e+02  2.23852754e-01  0.00000000e+00
   1.71750000e+04 -5.21522722e-01]]


In [6]:
nci_metrics

[[0.04848578275431983,
  773.9679790045641,
  0.19931066440993037,
  0.0,
  17182,
  -0.48854328964647725],
 [0.050820721524724444,
  747.5841679448183,
  0.1977177799565644,
  0.0,
  17178,
  -0.5379196107509626],
 [0.05054101647988111,
  794.7316331262278,
  0.19111923284428411,
  0.0,
  17203,
  -0.4886860630178973],
 [0.05050342772399721,
  751.3292127222846,
  0.21092176598286275,
  0.0,
  17186,
  -0.5282899442087738],
 [0.049424840044524125,
  745.7364384377303,
  0.2238527535037165,
  0.0,
  17175,
  -0.5215227220495245],
 [0.0492768791989006,
  801.6169786444743,
  0.26265647275564974,
  0.0,
  17258,
  -0.44093672917165677],
 [0.049734136635211514,
  752.494144444716,
  0.20130257580656644,
  0.0,
  17182,
  -0.5233828855996331],
 [0.050616742596326994,
  750.5048791410217,
  0.18177627892604317,
  0.0,
  17173,
  -0.5327162859895831],
 [0.04937377134098445,
  769.1254067315149,
  0.22540195776453614,
  0.0,
  17185,
  -0.49635290144180766],
 [0.0483544754268019,
  737.505804

In [8]:
!mkdir ../../../data_preprocessed/grn_analysis_output/grns_features_output

In [9]:
import pickle

# Example array

# Save to pickle file
with open("../../../data_preprocessed/grn_analysis_output/grns_features_output/nci_metrics.pkl", "wb") as f:
    pickle.dump(nci_metrics, f)